# UCS420 Lab Assignment 9 — NLP Text Preprocessing
Enter your roll number and name, then run all cells.

In [28]:
ROLL_NO = 1024170112          # <-- your full roll number
NAME = "Anantpreet Kaur"            # <-- your name

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
assert ROLL_NO > 0 and NAME, "Enter your roll number and name first"

### Dataset generator — do not modify

In [20]:
import random, hashlib
import pandas as pd

def generate_dataset(roll_no):
    """Return (DataFrame of tickets, focus ticket IDs, fingerprint) for a roll number. Do not modify."""
    rng = random.Random(int(roll_no))

    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }

    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)

    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint

### Preprocessing pipeline — do not modify

In [21]:
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def is_punct(token):
    """A token is punctuation if EVERY character in it is in string.punctuation (e.g. '.', '...', '``', "''")."""
    return all(ch in string.punctuation for ch in token)

def process_ticket(text):
    """The official pipeline. Returns sentences, lowercase tokens, punctuation, stopwords and final tokens."""
    sentences = sent_tokenize(text)
    tokens = [t.lower() for s in sentences for t in word_tokenize(s)]
    punct = [t for t in tokens if is_punct(t)]
    stops = [t for t in tokens if t in STOP]
    final = [t for t in tokens if not is_punct(t) and t not in STOP]
    return {"sentences": sentences, "tokens": tokens, "punct": punct, "stops": stops, "final": final}

In [22]:
df, FOCUS, FINGERPRINT = generate_dataset(ROLL_NO)
print(f"Roll number: {ROLL_NO}   Name: {NAME}   Fingerprint: {FINGERPRINT}   Tickets: {len(df)}\n")
print(df.to_string(index=False))
df.to_csv(f"{ROLL_NO}_generated_dataset.csv", index=False)

Roll number: 1024170112   Name: Anantpreet Kaur   Fingerprint: 41ECC52F   Tickets: 14

Ticket_ID                                                                     Ticket_Text
      T01         I couldn't connect to the LMS after resetting my password. Please help.
      T02                                     The state-of-the-art lab (Lab-3) is closed.
      T03  I was locked out of the Python since yesterday. The same problem occurs again.
      T04               I cannot access the lms while uploading the file. This is urgent.
      T05                                   The email was working yesterday. Please help.
      T06  Printer has stopped syncing when connected to campus Wi-Fi. Please resolve it.
      T07 LMS has stopped syncing after resetting my password. Is there another solution?
      T08                                          Only 2 of 30 PCs boot after 3.5 hours.
      T09                             The LMS is not working. I have tried several times!
      T10    

---
## Q1. Preprocessing pipeline
Run process_ticket() on every ticket and show one table with, per ticket: sentences, tokens, punctuation tokens, stopwords, unique tokens and final tokens, plus a TOTAL row.

In [29]:
# Your code

# Q1. Preprocessing

results = []

for i, row in df.iterrows():
    ticket_id = row["Ticket_ID"]
    text = row["Ticket_Text"]

    result = process_ticket(text)

    results.append({
        "Ticket_ID": ticket_id,
        "Sentences": len(result["sentences"]),
        "Tokens": len(result["tokens"]),
        "Punctuation Tokens": len(result["punct"]),
        "Stopwords": len(result["stops"]),
        "Unique Tokens": len(set(result["tokens"])),
        "Final Tokens": len(result["final"])
    })

table = pd.DataFrame(results)

total = {
    "Ticket_ID": "TOTAL",
    "Sentences": table["Sentences"].sum(),
    "Tokens": table["Tokens"].sum(),
    "Punctuation Tokens": table["Punctuation Tokens"].sum(),
    "Stopwords": table["Stopwords"].sum(),
    "Unique Tokens": "",
    "Final Tokens": table["Final Tokens"].sum()
}

table = pd.concat([table, pd.DataFrame([total])], ignore_index=True)

print(table.to_string(index=False))

# Display tokens for one selected ticket
selected_text = df.iloc[0]["Ticket_Text"]
selected_result = process_ticket(selected_text)

print("\nSelected Ticket:", df.iloc[0]["Ticket_ID"])
print("Ticket Text:", selected_text)
print("Tokens:", selected_result["tokens"])
print("Punctuation Tokens:", selected_result["punct"])
print("Stopwords:", selected_result["stops"])
print("Final Tokens:", selected_result["final"])


Ticket_ID  Sentences  Tokens  Punctuation Tokens  Stopwords Unique Tokens  Final Tokens
      T01          2      15                   2          5            14             8
      T02          1       9                   3          2             9             4
      T03          2      16                   2          8            14             6
      T04          2      15                   2          8            13             5
      T05          2       9                   2          2             8             5
      T06          2      14                   2          4            13             8
      T07          2      14                   2          5            14             7
      T08          1      10                   1          3            10             6
      T09          2      12                   2          5            12             5
      T10          2       7                   2          3             6             2
      T11          2      16    

**Observe:** Which final tokens in your data are not real words (e.g. n't, 's, ca)? Where do they come from?

*Your answer:*
 Tokens like "n't,'s,ca" are not complete words , and they come from short froms like do not is written as don't , could not is written as couldn't."'s"
 comes from worls like it's, her's etc. the tokenizer splits these into smaller parts and they will remain in the list if not removed.



---
## Q2. Regular expressions
On your raw tickets, use re.findall to extract: words longer than 5 letters; all numbers (keep 95.5 and 2.0.1 whole); capitalized words; words starting with a vowel. Then use re.sub to replace emails with <EMAIL>, URLs with <URL> and phone numbers with <PHONE>. Your patterns must also work on TEST_LINES (given in the notebook).

In [30]:
TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]

# Your code

import re


texts = df["Ticket_Text"].tolist()

for i, text in enumerate(texts):
    print("\nTicket", i + 1)


    long_words = re.findall(r'\b[A-Za-z]{6,}\b', text)
    print("Long words:", long_words)


    numbers = re.findall(r'\b\d+(?:\.\d+)*\b', text)
    print("Numbers:", numbers)


    capitalized_words = re.findall(r'\b[A-Z][a-zA-Z]*\b', text)
    print("Capitalized words:", capitalized_words)


    vowel_words = re.findall(r'\b[AEIOUaeiou][a-zA-Z]*\b', text)
    print("Vowel words:", vowel_words)


    cleaned = re.sub(
        r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b',
        '<EMAIL>', text
    )


    cleaned = re.sub(
        r'https?://\S+|www\.\S+',
        '<URL>', cleaned
    )


    cleaned = re.sub(
        r'(?<!\w)(?:\+?\d[\d\s().-]{7,}\d)(?!\w)',
        '<PHONE>', cleaned
    )

    print("Replaced text:", cleaned)



Ticket 1
Long words: ['couldn', 'connect', 'resetting', 'password', 'Please']
Numbers: []
Capitalized words: ['I', 'LMS', 'Please']
Vowel words: ['I', 'after']
Replaced text: I couldn't connect to the LMS after resetting my password. Please help.

Ticket 2
Long words: ['closed']
Numbers: ['3']
Capitalized words: ['The', 'Lab']
Vowel words: ['of', 'art', 'is']
Replaced text: The state-of-the-art lab (Lab-3) is closed.

Ticket 3
Long words: ['locked', 'Python', 'yesterday', 'problem', 'occurs']
Numbers: []
Capitalized words: ['I', 'Python', 'The']
Vowel words: ['I', 'out', 'of', 'occurs', 'again']
Replaced text: I was locked out of the Python since yesterday. The same problem occurs again.

Ticket 4
Long words: ['cannot', 'access', 'uploading', 'urgent']
Numbers: []
Capitalized words: ['I', 'This']
Vowel words: ['I', 'access', 'uploading', 'is', 'urgent']
Replaced text: I cannot access the lms while uploading the file. This is urgent.

Ticket 5
Long words: ['working', 'yesterday', 'Plea

**Observe:** How many of your capitalized words are capitalized only because they start a sentence?

*Your answer:* Some capitalized words are capitalized only because they appear at the beginning of a sentence. These are detected by the regex pattern, even though they are not proper nouns. The exact number depends on the generated ticket dataset.

---
## Q3. Custom tokenizer
Write my_tokenize(text) with one RegexpTokenizer pattern that keeps contractions (isn't), hyphenated words (Wi-Fi, state-of-the-art), decimals and versions (95.5, 2.0.1), emails, URLs and phone numbers as single tokens. Run it on TEST_LINES and on your tickets, and compare with word_tokenize.

In [32]:
# Your code
from nltk.tokenize import RegexpTokenizer, word_tokenize

pattern = r"https?://\S+|www\.\S+|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}|\+?\d[\d\s().-]{7,}\d|\d+(?:\.\d+)+|[A-Za-z]+(?:-[A-Za-z]+)+|[A-Za-z]+(?:'[A-Za-z]+)+|[A-Za-z]+|\d+|[^\w\s]"

tokenizer = RegexpTokenizer(pattern)

def my_tokenize(text):
    return tokenizer.tokenize(text)



print("TEST LINES")
for line in TEST_LINES:
    print("\nText:", line)
    print("My tokenizer:", my_tokenize(line))
    print("word_tokenize:", word_tokenize(line))



print("\nTICKETS")
for i, text in enumerate(df["Ticket_Text"]):
    print("\nTicket", i + 1)
    print("My tokenizer:", my_tokenize(text))
    print("word_tokenize:", word_tokenize(text))



print("\nTHREE DIFFERENCES")
count = 0

for text in list(TEST_LINES) + df["Ticket_Text"].tolist():
    mine = my_tokenize(text)
    normal = word_tokenize(text)

    if mine != normal:
        print("\nText:", text)
        print("My tokenizer:", mine)
        print("word_tokenize:", normal)
        count += 1

    if count == 3:
        break
# question's answer
count = 0

for text in df["Ticket_Text"]:
    if my_tokenize(text) != word_tokenize(text):
        count += 1

print("Tickets tokenized differently:", count)

TEST LINES

Text: Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.
My tokenizer: ['Mail', 'it.helpdesk@thapar.edu', 'or', 'a.b-c@dept.univ.ac.in', ';', 'visit', 'https://lms.thapar.edu/help', 'or', 'www.thapar.edu/it.']
word_tokenize: ['Mail', 'it.helpdesk', '@', 'thapar.edu', 'or', 'a.b-c', '@', 'dept.univ.ac.in', ';', 'visit', 'https', ':', '//lms.thapar.edu/help', 'or', 'www.thapar.edu/it', '.']

Text: Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.
My tokenizer: ['Call', '+91 98765 43210', ',', '+91-98765-43210', 'or', '0175-239-3201', '.', 'Version', '2.0.1', 'is', '95.5', '%', 'done', ';', 'fee', 'Rs', '1', ',', '250', '.']
word_tokenize: ['Call', '+91', '98765', '43210', ',', '+91-98765-43210', 'or', '0175-239-3201', '.', 'Version', '2.0.1', 'is', '95.5', '%', 'done', ';', 'fee', 'Rs', '1,250', '.']

Text: The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't

**Observe:** How many of your tickets are tokenized differently by my_tokenize and word_tokenize?

*Your answer:* 6


---
## Q4. Stemming and lemmatization
On your unique final tokens, compare PorterStemmer, LancasterStemmer, a RegexpStemmer with a suffix rule you choose after looking at your own words, and WordNetLemmatizer with POS tags from nltk.pos_tag. Show a table of the words where the outputs differ.

In [33]:
# Your code
import nltk
from nltk import pos_tag
from nltk.stem import PorterStemmer, LancasterStemmer
from nltk.stem import RegexpStemmer, WordNetLemmatizer

nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')


all_tokens = []

for text in df["Ticket_Text"]:
    result = process_ticket(text)
    all_tokens.extend(result["final"])

words = sorted(set(all_tokens))


porter = PorterStemmer()
lancaster = LancasterStemmer()
regexp = RegexpStemmer(r'ing$|ed$|es$')
lemmatizer = WordNetLemmatizer()


tags = pos_tag(words)

print("Word | Porter | Lancaster | Regexp | Lemmatizer")
print("-" * 65)

for word, tag in tags:

    if tag.startswith('J'):
        pos = 'a'
    elif tag.startswith('V'):
        pos = 'v'
    elif tag.startswith('N'):
        pos = 'n'
    elif tag.startswith('R'):
        pos = 'r'
    else:
        pos = 'n'

    p = porter.stem(word)
    l = lancaster.stem(word)
    r = regexp.stem(word)
    lemma = lemmatizer.lemmatize(word, pos=pos)


    if len(set([p, l, r, lemma])) > 1:
        print(word, "|", p, "|", l, "|", r, "|", lemma)


print("\nRegexp Stemmer justification:")
print("The suffixes ing, ed and es are removed from the ends of words.")
print("These suffixes occur in words such as playing, played and wishes.")

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Word | Porter | Lancaster | Regexp | Lemmatizer
-----------------------------------------------------------------
another | anoth | anoth | another | another
campus | campu | camp | campus | campus
changed | chang | chang | chang | change
closed | close | clos | clos | closed
file | file | fil | file | file
hours | hour | hour | hours | hour
lms | lm | lms | lms | lm
occurs | occur | occ | occurs | occur
open | open | op | open | open
pcs | pc | pcs | pcs | pc
please | pleas | pleas | please | please
printer | printer | print | printer | printer
resetting | reset | reset | resett | reset
resolve | resolv | resolv | resolve | resolve
roll | roll | rol | roll | roll
running | run | run | runn | run
server | server | serv | server | server
several | sever | sev | several | several
since | sinc | sint | since | since
slowly | slowli | slow | slowly | slowly
solution | solut | solv | solution | solution
stopped | stop | stop | stopp | stop
syncing | sync | synt | sync | sync
times | time | 

**Observe:** Give one over-stemming and one under-stemming example from your table.

*Your answer:* **Over-stemming:** The word `solution` becomes `solv` with Lancaster Stemmer. The stem is shortened too much and becomes less recognizable.

**Under-stemming:** The words `changed → chang` and `closed → close` with Porter Stemmer are not reduced to the same stem, even though both are past-tense forms.

---
## Q5. Corpus statistics
Compute tokens, unique tokens and TTR for S1 (all tokens), S2 (no punctuation), S3 (final tokens) and S4 (S3 after Porter). Print the top 10 words of S1 and S3, plot log(rank) vs log(frequency) for S3, and print the top 5 bigrams of S1 and S3.

In [34]:
# Your code
import string
import math
import nltk
import matplotlib.pyplot as plt
from collections import Counter
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.util import bigrams

S1 = []

for text in df["Ticket_Text"]:
    S1.extend(word_tokenize(text.lower()))

S2 = [word for word in S1 if word not in string.punctuation]

S3 = []

for text in df["Ticket_Text"]:
    result = process_ticket(text)
    S3.extend([word.lower() for word in result["final"]])

porter = PorterStemmer()
S4 = [porter.stem(word) for word in S3]

sets = {
    "S1 - All tokens": S1,
    "S2 - Without punctuation": S2,
    "S3 - Final tokens": S3,
    "S4 - Porter stemming": S4
}

print("Set | Total tokens | Unique tokens | TTR")
print("-" * 55)

for name, words in sets.items():
    total = len(words)
    unique = len(set(words))
    ttr = unique / total if total > 0 else 0
    print(name, "|", total, "|", unique, "|", round(ttr, 4))

print("\nTop 10 words in S1:")
print(Counter(S1).most_common(10))

print("\nTop 10 words in S3:")
print

Set | Total tokens | Unique tokens | TTR
-------------------------------------------------------
S1 - All tokens | 175 | 87 | 0.4971
S2 - Without punctuation | 147 | 81 | 0.551
S3 - Final tokens | 80 | 55 | 0.6875
S4 - Porter stemming | 80 | 54 | 0.675

Top 10 words in S1:
[('.', 22), ('the', 14), ('i', 7), ('is', 6), ('lms', 4), ('my', 4), ("n't", 3), ('after', 3), ('please', 3), ('not', 3)]

Top 10 words in S3:


<function print(*args, sep=' ', end='\n', file=None, flush=False)>

**Observe:** Why does TTR change from S1 to S4 in your data?

*Your answer:*TTR decreases from S1 to S4 because stemming converts different forms of words into the same stem. This reduces the number of unique tokens while the total number of tokens remains almost the same. Therefore, the vocabulary becomes smaller and the Type-Token Ratio decreases.